# class 4

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
import time

from IPython.display import display, HTML

pd.set_option("display.max_columns", 50)
pd.set_option("display.precision", 2)


def rtl(text):
    """Display Hebrew/RTL text nicely from code cells."""
    display(HTML(f"""
    <div dir="rtl" style="text-align:right; font-family:Arial, 'Noto Sans Hebrew', sans-serif; font-size:16px; line-height:1.7">
        {text}
    </div>
    """))


def show_title(title):
    rtl(f"<h3>{title}</h3>")


rtl("הספריות נטענו בהצלחה. נתחיל בתרגול.")

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# יצירת הדאטה לתרגול

ניצור דאטה מלאכותי של עסק שמוכר מוצרים טכנולוגיים.

הדאטה יכלול:

- מוצר
- קטגוריה
- אזור
- ערוץ מכירה
- כמות
- מחיר
- הנחה
- תאריך
- גיל לקוח
- דירוג לקוח

נכניס בכוונה גם ערכים חסרים וחריגים כדי לתרגל ניקוי נתונים ו־EDA.

</div>

In [3]:
rng = np.random.default_rng(42)

n = 120

product_names = np.array([
    "Laptop", "Keyboard", "Mouse", "Monitor", "Headphones", "Webcam"
])

categories = {
    "Laptop": "Computer",
    "Keyboard": "Peripheral",
    "Mouse": "Peripheral",
    "Monitor": "Display",
    "Headphones": "Audio",
    "Webcam": "Video",
}

base_prices = {
    "Laptop": 3500,
    "Keyboard": 250,
    "Mouse": 120,
    "Monitor": 1100,
    "Headphones": 450,
    "Webcam": 320,
}

regions = np.array(["North", "Center", "South", "Jerusalem"])
channels = np.array(["Online", "Store", "Partner"])
dates = pd.date_range("2026-01-01", periods=60, freq="D")

df = pd.DataFrame({
    "id": np.arange(1, n + 1),
    "product": rng.choice(product_names, n, p=[0.18, 0.20, 0.22, 0.14, 0.16, 0.10]),
    "region": rng.choice(regions, n, p=[0.20, 0.45, 0.20, 0.15]),
    "channel": rng.choice(channels, n, p=[0.55, 0.35, 0.10]),
    "quantity": rng.integers(1, 6, n),
    "discount": rng.choice([0, 0.05, 0.10, 0.15, np.nan], n, p=[0.45, 0.20, 0.18, 0.12, 0.05]),
    "date": rng.choice(dates, n),
    "customer_age": rng.integers(18, 70, n),
    "rating": np.round(rng.normal(4.1, 0.7, n), 1),
})

df["price"] = df["product"].map(base_prices).astype(float)
df["price"] = (df["price"] * rng.normal(1, 0.05, n)).round(2)
df["category"] = df["product"].map(categories)

# Ratings should be between 1 and 5
df["rating"] = df["rating"].clip(1, 5)

# Add missing values intentionally
df.loc[rng.choice(df.index, size=6, replace=False), "rating"] = np.nan

# Add one clear outlier intentionally
df.loc[5, "quantity"] = 30

df["date"] = pd.to_datetime(df["date"])


display(df.head())

,id,product,region,channel,quantity,discount,date,customer_age,rating,price,category
0,1,Headphones,Center,Store,1,0.10,2026-02-25,63,3.9,475.56,Audio
1,2,Mouse,Center,Store,5,0.00,2026-03-01,42,4.0,114.47,Peripheral
2,3,Headphones,North,Online,5,0.00,2026-02-27,69,3.3,469.24,Audio
3,4,Monitor,Center,Online,1,0.00,2026-02-03,57,4.5,1135.18,Display
4,5,Laptop,North,Store,4,0.05,2026-01-13,51,4.5,3577.45,Computer


<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# חלק א׳ — שאלות תיאורטיות לפתיחה

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## שאלה 1 — תכנות רגיל מול למידת מכונה

נניח שאנחנו רוצים להעריך מחיר שכירות של דירה לפי שטח הדירה.

ענו:

1. איך היינו פותרים זאת בגישה של תכנות רגיל?
2. איך היינו פותרים זאת בגישה של למידת מכונה?
3. מה היתרון של למידת מכונה כאשר הקשר בין המשתנים מורכב?

</div>

<div class="todo" dir="rtl" style="direction: rtl; text-align: right;">

### מקום לתשובה

כתבו תשובה מילולית קצרה. נסו להשתמש במבנה:

**בתכנות רגיל:** ...  
**בלמידת מכונה:** ...  
**היתרון:** ...

</div>

</div>

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## שאלה 2 — סביבת עבודה מסודרת

למה חשוב לעבוד עם סביבת פייתון מסודרת כמו `venv`, `conda`?

התייחסו למושגים:

- Reproducibility
- Isolation
- Portability
- Reliability

</div>

<div class="todo" dir="rtl" style="direction: rtl; text-align: right;">

### מקום לתשובה

כתבו תשובה במילים שלכם. מומלץ לתת משפט אחד לכל מושג.

</div>

</div>

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## שאלה 3 — רשימות רגילות מול NumPy Arrays

מה ההבדל בין `list` רגילה בפייתון לבין `np.array`?

התייחסו ל:

1. סוגי נתונים
2. מהירות
3. פעולות וקטוריות
4. שימושים ב־Data Science

</div>

<div class="todo" dir="rtl" style="direction: rtl; text-align: right;">

### מקום לתשובה

כתבו תשובה קצרה. נסו להסביר למה NumPy נפוץ בעבודה עם נתונים מספריים.

</div>

</div>

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# חלק ב׳ — NumPy

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 1 — יצירת מערכים ו־reshape

צרו מערך NumPy עם המספרים מ־1 עד 12.

לאחר מכן:

1. הציגו את הצורה שלו בעזרת `.shape`.
2. שנו אותו למטריצה בגודל 3x4.
3. חשבו סכום של כל שורה.
4. חשבו ממוצע של כל עמודה.

</div>

</div>

In [ ]:
# TODO: צרו מערך עם המספרים 1 עד 12
arr = None

# TODO: הדפיסו את צורת המערך

# TODO: הפכו את המערך למטריצה בגודל 3x4
matrix = None

# TODO: חשבו סכום לכל שורה
row_sums = None

# TODO: חשבו ממוצע לכל עמודה
col_means = None

print("arr:")
print(arr)
print("matrix:")
print(matrix)
print("row_sums:", row_sums)
print("col_means:", col_means)

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 2 — Broadcasting

יש לנו מטריצה שבה כל שורה מייצגת אזור מכירה וכל עמודה מייצגת מוצר.

יש לנו גם וקטור הנחות לפי מוצר.

חשבו את המכירות לאחר הנחה בלי להשתמש בלולאה.

</div>

</div>

In [ ]:
sales = np.array([
    [10000, 2000, 1200, 5000],
    [15000, 3000, 1800, 6500],
    [8000,  1500, 900,  4000],
])

discount_by_product = np.array([0.10, 0.05, 0.00, 0.15])

# TODO: חשבו את המכירות לאחר הנחה
net_sales = None

print(net_sales)

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## שאלה תיאורטית — Broadcasting

מה צריך להתקיים כדי ש־Broadcasting יעבוד?

</div>

<div class="todo" dir="rtl" style="direction: rtl; text-align: right;">
כתבו תשובה קצרה. התייחסו לצורה של מטריצה בגודל 3x4 ולווקטור באורך 4.
</div>

</div>

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# חלק ג׳ — Pandas: בדיקה ראשונית של הדאטה

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 3 — מבט ראשוני

בדקו:

1. כמה שורות ועמודות יש בדאטה?
2. מהם טיפוסי הנתונים של העמודות?
3. האם יש ערכים חסרים?
4. הציגו סטטיסטיקה תיאורית לעמודות המספריות.
5. הציגו סטטיסטיקה תיאורית גם לעמודות קטגוריאליות.

</div>

</div>

In [ ]:
# TODO: הציגו את צורת הדאטה

# TODO: הציגו טיפוסי נתונים

# TODO: בדקו ערכים חסרים

# TODO: סטטיסטיקה תיאורית לעמודות מספריות

# TODO: סטטיסטיקה תיאורית לכל העמודות

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## שאלה תיאורטית — למה מבצעים EDA?

למה חשוב לבצע Exploratory Data Analysis לפני בניית מודל?


</div>

<div class="todo" dir="rtl" style="direction: rtl; text-align: right;">
מקום לתשובה.
</div>

</div>

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# חלק ד׳ — בחירה וסינון עם Pandas

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 4 — בחירת עמודות וסינון שורות

בצעו:

1. הציגו רק את העמודות `product`, `region`, `quantity`, `price`.
2. הציגו רק עסקאות מאזור `Center`.
3. הציגו עסקאות שבהן הכמות גדולה מ־3.
4. הציגו עסקאות מאזור `Center` שבהן הכמות גדולה מ־3.
5. השתמשו ב־`.loc` כדי להציג רק את העמודות `product`, `quantity`, `price` עבור העסקאות האלו.

</div>

</div>

In [ ]:
# TODO 1: בחירת עמודות

# TODO 2: עסקאות מאזור Center

# TODO 3: עסקאות שבהן quantity > 3

# TODO 4-5: Center וגם quantity > 3, עם בחירת עמודות

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## שאלה תיאורטית — `loc` מול `iloc`

מה ההבדל בין `df.loc[...]` לבין `df.iloc[...]`?

</div>

<div class="todo" dir="rtl" style="direction: rtl; text-align: right;">
מקום לתשובה.
</div>

</div>

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# חלק ה׳ — ניקוי נתונים ויצירת עמודות חדשות

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 5 — טיפול בערכים חסרים וחישוב הכנסה

בצעו:

1. צרו עותק בשם `clean_df`.
2. מלאו הנחות חסרות ב־0.
3. מלאו דירוגים חסרים לפי ממוצע הדירוג של אותו מוצר.
4. צרו עמודת `revenue` לפי הנוסחה:

`revenue = quantity * price * (1 - discount)`

5. עגלו את ההכנסה לשתי ספרות אחרי הנקודה.

</div>

</div>

In [ ]:
# TODO: צרו עותק
clean_df = None

# TODO: מלאו הנחות חסרות ב-0

# TODO: מלאו דירוגים חסרים לפי ממוצע מוצר

# TODO: צרו עמודת revenue

# TODO: בדקו שאין ערכים חסרים בעמודות הרלוונטיות

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 6 — זיהוי חריגים בעמודת quantity

השתמשו בשיטת IQR:

- חשבו Q1.
- חשבו Q3.
- חשבו IQR.
- הגדירו גבול תחתון וגבול עליון.
- הציגו את השורות החריגות.

</div>

</div>

In [ ]:
# TODO: חשבו Q1, Q3, IQR
q1 = None
q3 = None
iqr = None

# TODO: הגדירו גבולות
lower_bound = None
upper_bound = None

# TODO: סננו חריגים
outliers = None

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("lower_bound:", lower_bound)
print("upper_bound:", upper_bound)

display(outliers)

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# חלק ו׳ — groupby וניתוח עסקי

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 7 — מוצרים מובילים

מצאו לכל מוצר:

1. סך הכנסה.
2. מספר עסקאות.
3. דירוג ממוצע.

לאחר מכן מיינו לפי הכנסה בסדר יורד.

</div>

</div>

In [ ]:
# TODO: groupby לפי product
# שמרו את התוצאה במשתנה top_products

top_products = None

display(top_products)

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 8 — ניתוח לפי אזור וערוץ מכירה

ענו:

1. מה ההכנסה הכוללת בכל אזור?
2. מה ההכנסה הממוצעת לעסקה בכל ערוץ מכירה?
3. באיזה שילוב של אזור וערוץ הייתה ההכנסה הגבוהה ביותר?

</div>

</div>

In [ ]:
# TODO 1: הכנסה כוללת לפי אזור
revenue_by_region = None

# TODO 2: הכנסה ממוצעת לפי ערוץ מכירה
avg_revenue_by_channel = None

# TODO 3: הכנסה לפי שילוב אזור וערוץ
region_channel = None

print("revenue_by_region")
display(revenue_by_region)

print("avg_revenue_by_channel")
display(avg_revenue_by_channel)

print("region_channel")
display(region_channel)

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# חלק ז׳ — SQL דרך SQLite

בשיעור הקורס מופיע גם SQL. כדי לא לדרוש התקנה של שרת MySQL, נשתמש כאן ב־SQLite זמני מתוך פייתון.

הרעיון מבחינת שאילתות בסיסיות דומה:

- `SELECT`
- `FROM`
- `WHERE`
- `GROUP BY`
- `ORDER BY`
- `LIMIT`

</div>

In [ ]:
# יצירת מסד נתונים זמני בזיכרון
conn = sqlite3.connect(":memory:")

# ודאו ש-clean_df קיים מהחלק הקודם
clean_df.to_sql("transactions", conn, if_exists="replace", index=False)

rtl("הטבלה transactions נוצרה במסד נתונים זמני בזיכרון.")

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 9 — SQL בסיסי

כתבו שאילתה שמחזירה:

1. רק את העמודות `product`, `region`, `quantity`, `revenue`.
2. רק עסקאות שבהן `quantity > 3`.
3. מיון לפי `revenue` מהגבוה לנמוך.
4. רק 10 השורות הראשונות.

</div>

</div>

In [ ]:
query = """
-- TODO: כתבו כאן את השאילתה
"""

sql_result = pd.read_sql_query(query, conn)
display(sql_result)

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 10 — SQL מול Pandas

כתבו שאילתת SQL שמחזירה לכל מוצר:

- סך הכנסה
- מספר עסקאות
- דירוג ממוצע

מיינו לפי הכנסה בסדר יורד.

לאחר מכן השוו לתוצאה שקיבלנו ב־Pandas.

</div>

</div>

In [ ]:
query = """
-- TODO: כתבו כאן שאילתת GROUP BY
"""

sql_top_products = pd.read_sql_query(query, conn)
display(sql_top_products)

# השוואה לתוצאה מ-Pandas
# display(top_products.reset_index())

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## שאלה תיאורטית — מתי SQL ומתי Pandas?

מתי עדיף להשתמש ב־SQL, ומתי עדיף להשתמש ב־Pandas?

</div>

<div class="todo" dir="rtl" style="direction: rtl; text-align: right;">
מקום לתשובה.
</div>

</div>

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# חלק ח׳ — EDA וגרפים

<div class="note" dir="rtl" style="direction: rtl; text-align: right;">
כותרות הגרפים מסומנות באנגלית כדי למנוע בעיות תצוגה של עברית בתוך תמונות Matplotlib.
</div>

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 11 — התפלגות הכנסות

צרו היסטוגרמה של עמודת `revenue`.

שאלות לדיון:

1. האם רוב העסקאות קטנות או גדולות?
2. האם יש עסקאות חריגות?
3. האם ההתפלגות סימטרית?

</div>

</div>

In [ ]:
# TODO: צרו היסטוגרמה של revenue
plt.figure(figsize=(8, 4))
# clean_df["revenue"].hist(bins=20)
plt.title("Revenue Distribution")
plt.xlabel("Revenue")
plt.ylabel("Number of Transactions")
plt.show()

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 12 — קשר בין כמות להכנסה

צרו scatter plot בין `quantity` לבין `revenue`.

שאלות לדיון:

1. האם ככל שהכמות עולה, ההכנסה עולה?
2. האם יש נקודות חריגות?
3. האם המחיר של המוצר משפיע על הקשר?

</div>

</div>

In [ ]:
# TODO: צרו scatter plot של quantity מול revenue
plt.figure(figsize=(8, 4))
# plt.scatter(clean_df["quantity"], clean_df["revenue"], alpha=0.7)
plt.title("Quantity vs Revenue")
plt.xlabel("Quantity")
plt.ylabel("Revenue")
plt.show()

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 13 — הכנסות לפי מוצר

צרו גרף עמודות של סך ההכנסות לפי מוצר.

</div>

</div>

In [ ]:
# TODO: צרו bar plot של הכנסות לפי מוצר
plt.figure(figsize=(8, 4))
# top_products["revenue"].plot(kind="bar")
plt.title("Revenue by Product")
plt.xlabel("Product")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.show()

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 14 — מטריצת קורלציה

בדקו קורלציות בין המשתנים המספריים:

- `quantity`
- `price`
- `discount`
- `customer_age`
- `rating`
- `revenue`

לאחר מכן הציגו heatmap פשוט בעזרת Matplotlib.

</div>

</div>

In [ ]:
numeric_cols = ["quantity", "price", "discount", "customer_age", "rating", "revenue"]

# TODO: חשבו מטריצת קורלציה
corr = None

# TODO: הציגו את מטריצת הקורלציה ואת ה-heatmap
# display(corr)

plt.figure(figsize=(7, 5))
# plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## תרגיל 15 — Pivot Table ו־Heatmap

צרו טבלת Pivot שבה:

- השורות הן `region`
- העמודות הן `channel`
- הערכים הם סכום `revenue`

לאחר מכן הציגו אותה כ־heatmap.

</div>

</div>

In [ ]:
# TODO: צרו Pivot Table
pivot = None

# display(pivot)

# TODO: הציגו Heatmap של הטבלה
plt.figure(figsize=(7, 4))
# plt.imshow(pivot.values)
plt.colorbar(label="Revenue")
# plt.xticks(range(len(pivot.columns)), pivot.columns)
# plt.yticks(range(len(pivot.index)), pivot.index)
plt.title("Revenue by Region and Channel")
plt.tight_layout()
plt.show()

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# חלק ט׳ — קידוד קטגוריאלי

מודלים רבים של Machine Learning צריכים קלט מספרי. לכן משתנים כמו `product`, `region`, `channel`, `category` צריכים לעבור המרה לייצוג מספרי.

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

## שאלה תיאורטית — One-Hot Encoding מול Dummy Encoding

מה ההבדל בין One-Hot Encoding לבין Dummy Encoding?

</div>

<div class="todo" dir="rtl" style="direction: rtl; text-align: right;">
מקום לתשובה.
</div>

</div>

In [ ]:
# TODO: צרו One-Hot Encoding לעמודת channel
one_hot_channel = None

# TODO: צרו Dummy Encoding לעמודת channel עם drop_first=True
dummy_channel = None

# TODO: הציגו את שתי התוצאות

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# חלק י׳ — משימה מסכמת

<div class="question" dir="rtl" style="direction: rtl; text-align: right;">

בנו ניתוח קצר שמחזיר תשובות לשאלות:

1. מה המוצר עם ההכנסה הגבוהה ביותר?
2. מה האזור עם ההכנסה הגבוהה ביותר?
3. מה ערוץ המכירה עם ההכנסה הממוצעת הגבוהה ביותר?
4. כמה ערכים חריגים יש בעמודת `quantity`?
5. האם לדעתכם כדאי להסיר את החריגים? נמקו.
6. איזו תובנה עסקית אחת אפשר להציג למנהל?

</div>

</div>

In [ ]:
# TODO: חשבו את התשובות למשימה המסכמת

best_product = None
best_region = None
best_channel = None
num_outliers = None

print("best_product:", best_product)
print("best_region:", best_region)
print("best_channel:", best_channel)
print("num_outliers:", num_outliers)

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# שאלות הרחבה אם נשאר זמן

1. צרו עמודה בשם `revenue_level` עם הערכים `Low`, `Medium`, `High`.
2. בדקו מה הדירוג הממוצע בכל רמת הכנסה.
3. נסו למלא דירוגים חסרים לפי ממוצע קטגוריה במקום ממוצע מוצר.
4. הציגו הכנסה לאורך זמן לפי תאריך.

</div>

In [ ]:
# TODO: הרחבה - הכנסה לאורך זמן
# revenue_by_date = clean_df.groupby("date")["revenue"].sum().sort_index()

# plt.figure(figsize=(8, 4))
# revenue_by_date.plot()
# plt.title("Revenue Over Time")
# plt.xlabel("Date")
# plt.ylabel("Revenue")
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

<div class="rtl" dir="rtl" style="direction: rtl; text-align: right;">

# סיכום

בתרגול עברנו על:

- שאלות תיאורטיות על תכנות רגיל מול למידת מכונה.
- NumPy ו־Broadcasting.
- Pandas לבדיקת נתונים, סינון וניקוי.
- טיפול בערכים חסרים וחריגים.
- groupby וניתוח עסקי.
- SQL בסיסי והשוואה ל־Pandas.
- EDA וגרפים.
- קידוד קטגוריאלי.

המסר המרכזי:  
**ב־Data Science, העבודה החשובה מתחילה מהבנת הנתונים — עוד לפני בניית המודל.**

</div>